# elbo-loss-sum-with-beta — worked example 3: Beta annealing over a warmup schedule

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `elbo-loss-sum-with-beta`.

**This is a worked example — read it, run each cell, and follow the reasoning.** It's study material, so there's nothing to submit here. Delta Drills hands you a hands-on version to complete yourself as you get comfortable with the idea.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
import matplotlib.pyplot as plt

## Concept

_First time on this topic? Run the **Setup** cell above and skim it: every class and helper mentioned below is defined there. You don't need to have done any other drill first._

Many VAE recipes ramp `beta` linearly from 0 up to a target over the first few epochs (KL annealing) to avoid posterior collapse. The loss combinator is unchanged — `recon + beta * kl` — but `beta` is now a function of the training step. The composite loss therefore grows as the KL term is phased in.

## Worked solution

We compute the ELBO across a warmup schedule where `beta` ramps linearly from 0 to a target value.

1. Fix the two loss terms `recon` and `kl` (held constant here so the schedule effect is isolated).
2. For each step in the warmup window, the annealed weight is `beta_target * step / warmup`, clamped so it never exceeds the target. At step 0 it is 0; at `step >= warmup` it sits at the target.
3. Apply the same combinator `recon + beta_t * kl` at each step. Because `kl` is positive, the loss is non-decreasing in `beta_t`, so the trajectory rises monotonically until the schedule plateaus.
4. We print the per-step losses to show the ramp and verify the first equals `recon` (beta 0) and the last equals `recon + beta_target * kl`.

In [ ]:
import torch as t

t.manual_seed(2)
recon = t.tensor(1.2)
kl = t.tensor(0.6)

def annealed_losses(recon, kl, beta_target, warmup, n_steps):
    out = []
    for step in range(n_steps):
        beta_t = beta_target * min(step / warmup, 1.0)
        out.append(recon + beta_t * kl)
    return t.stack(out)

losses = annealed_losses(recon, kl, beta_target=4.0, warmup=5, n_steps=8)
print([round(float(v), 3) for v in losses])
print('starts at recon:', bool(t.isclose(losses[0], recon)))
print('ends at recon+4kl:', bool(t.isclose(losses[-1], recon + 4.0 * kl)))